# Pipeline de Parsing Estructural para RAG Jerárquico

En este notebook se implementa el primer paso del sistema RAG jerárquico: el parsing estructural del PDF.

El objetivo es transformar un documento PDF en una estructura jerárquica que permita:

- Identificar secciones y subsecciones
- Mantener trazabilidad por página
- Preparar el documento para chunking semántico posterior
- Facilitar la citación automática en respuestas del asistente

Este pipeline constituye el **Nivel 2 del sistema RAG**, encargado de reconstruir el contexto ampliado del documento.


In [9]:
from unstructured.partition.pdf import partition_pdf
from unstructured.documents.elements import Title, NarrativeText, Table, ListItem
import uuid
import json
from collections import defaultdict
import os

## Parsing estructural del documento

Se utiliza la función `partition_pdf` de la librería Unstructured para analizar el contenido del documento.

Este proceso identifica distintos tipos de elementos:

- Títulos de sección
- Texto narrativo
- Listas
- Tablas

El parser utiliza modelos de clasificación documental y heurísticas basadas en layout para segmentar el contenido.

El resultado es una lista ordenada de elementos estructurales del documento.


In [11]:
import os

PDF_PATH = "/home/pablo/Documentos/ictl-rag/rag-asistent/data/pdf/guia_gestion_ciberincidentes .pdf"

print("Existe el archivo:", os.path.exists(PDF_PATH))

elements = partition_pdf(
    filename=PDF_PATH,
    strategy="fast",
    infer_table_structure=True,
    extract_images_in_pdf=False,
    languages=["spa"]
)

print("Número de elementos:", len(elements))

Existe el archivo: True
Número de elementos: 1028


## Inspección de resultados del parsing

Se visualizan los primeros elementos detectados para validar la calidad del parsing.

Permite comprobar:

- Correcta detección de títulos
- Presencia de metadatos de página
- Orden lógico del documento

Esta validación es fundamental antes de continuar con el procesamiento estructural.

In [12]:
for el in elements[:20]:
    print(type(el), el.text[:120])
    print(el.metadata)
    print("-"*50)

<class 'unstructured.documents.elements.Title'> GUÍA NACIONAL DE NOTIFICACIÓN Y GESTIÓN DE CIBERINCIDENTES
--------------------------------------------------
<class 'unstructured.documents.elements.Text'> APROBADO POR EL CONSEJO NACIONAL DE CIBERSEGURIDAD EL DÍA 21 DE FEBRERO DE 2020
--------------------------------------------------
<class 'unstructured.documents.elements.Title'> ÍNDICE
--------------------------------------------------
<class 'unstructured.documents.elements.ListItem'> 1. INTRODUCCIÓN ...................................................................................................... 5
--------------------------------------------------
<class 'unstructured.documents.elements.ListItem'> 2. OBJETO DE LA PRESENTE GUÍA .......................................................................... 7
--------------------------------------------------
<class 'unstructured.documents.elements.ListItem'> 3. ALCANCE ................................................................

## Normalización de elementos estructurales

Los objetos generados por el parser se convierten en un formato JSON homogéneo.

Este paso permite:

- Desacoplar la estructura del parser original
- Generar identificadores únicos
- Estandarizar metadatos necesarios para RAG


In [13]:
def normalize_element(el, source_name):

    return {
        "element_id": str(uuid.uuid4()),
        "type": el.category,
        "text": (el.text or "").strip(),
        "page": el.metadata.page_number,
        "source": source_name
    }


## Filtrado del índice del documento

Los documentos institucionales suelen incluir un índice que replica los títulos de las secciones.

Estos elementos no forman parte del contenido semántico real, por lo que se eliminan para evitar:

- Generación de secciones duplicadas
- Ruido en la indexación semántica

In [14]:
def is_index_item(text):

    if text is None:
        return False

    text = text.strip()

    # patrón típico del índice
    if "..." in text:
        return True

    return False

## Reconstrucción jerárquica del documento

Se agrupan los elementos detectados en secciones estructurales.

Cada sección contiene:

- Título de sección
- Rango de páginas
- Lista ordenada de elementos

Esta estructura será utilizada para:

- Reconstrucción del contexto ampliado (Nivel 2 del RAG)
- Navegación semántica del documento


In [15]:
def build_sections(elements, source_name):

    sections = []
    current_section = None
    inside_index = False

    for el in elements:

        text = (el.text or "").strip()

        # Detectar inicio índice
        if text.upper() == "ÍNDICE":
            inside_index = True
            continue

        # Salir del índice
        if inside_index and isinstance(el, Title):
            inside_index = False

        # Ignorar índice
        if inside_index:
            continue

        norm = normalize_element(el, source_name)

        # Detectar título real de sección
        if isinstance(el, Title):

            if current_section:
                sections.append(current_section)

            current_section = {
                "section_id": str(uuid.uuid4()),
                "section_title": norm["text"],
                "page_start": norm["page"],
                "page_end": norm["page"],
                "elements": []
            }

        else:

            if current_section is None:
                current_section = {
                    "section_id": str(uuid.uuid4()),
                    "section_title": "INTRODUCCIÓN",
                    "page_start": norm["page"],
                    "page_end": norm["page"],
                    "elements": []
                }

            current_section["elements"].append(norm)

            if norm["page"]:
                current_section["page_end"] = norm["page"]

    if current_section:
        sections.append(current_section)

    return sections


In [16]:
SOURCE_NAME = os.path.basename(PDF_PATH)

sections = build_sections(elements, SOURCE_NAME)

print("Número de secciones:", len(sections))

Número de secciones: 465


## Validación estructural

Se revisan las primeras secciones generadas para comprobar:

- Correcta detección de títulos
- Coherencia del rango de páginas
- Número adecuado de elementos por sección


In [27]:
for s in sections[:10]:

    print("Sección:", s["section_title"])
    print("Páginas:", s["page_start"], "-", s["page_end"])
    print("Elementos:", len(s["elements"]))
    print()

Sección: GUÍA NACIONAL DE NOTIFICACIÓN Y GESTIÓN DE CIBERINCIDENTES
Páginas: 1 - 1
Elementos: 1

Sección: ANEXO 2. NOTIFICACIÓN EN EL ÁMBITO DEL SECTOR PÚBLICO ....................... 40
Páginas: 3 - 4
Elementos: 2

Sección: ANEXO 3. NOTIFICACIÓN EN EL ÁMBITO DEL SECTOR PRIVADO ....................... 41
Páginas: 4 - 4
Elementos: 3

Sección: DE CARÁCTER PARTICULAR AL ÁMBITO DE LAS INFRAESTRUCTURAS CRÍTICAS .................... 43
Páginas: 4 - 4
Elementos: 0

Sección: DE CARÁCTER PARTICULAR A LAS REDES MILITARES Y DE DEFENSA ....................................... 44
Páginas: 4 - 5
Elementos: 18

Sección: O BJ ET
Páginas: 5 - 7
Elementos: 14

Sección: ◼ Responsables de Seguridad de la Información (RSI), como Responsables
Páginas: 7 - 7
Elementos: 0

Sección: Delegados.
Páginas: 7 - 7
Elementos: 2

Sección: O BJ ET
Páginas: 7 - 7
Elementos: 0

Sección: ◼ CSIRT (Computer Security Incident Response Team).
Páginas: 7 - 7
Elementos: 0



## Generación de offsets de caracteres

Se calculan posiciones de inicio y fin de cada elemento dentro de su sección.

Estos offsets permiten:

- Reconstruir fragmentos de contexto ampliado
- Facilitar citaciones precisas
- Implementar ventanas de contexto dinámicas


In [18]:
def add_char_offsets(sections):

    for sec in sections:

        cursor = 0

        for el in sec["elements"]:

            text = el["text"]

            el["char_start"] = cursor
            el["char_end"] = cursor + len(text)

            cursor += len(text) + 1

    return sections


In [19]:
sections = add_char_offsets(sections)


## Preparación para chunking semántico

Se transforma la estructura jerárquica en una lista plana de elementos.

Este formato será utilizado para:

- Generación de embeddings
- Indexación vectorial
- Recuperación semántica (Nivel 1 del RAG)

Cada elemento conserva metadatos de su sección original.


In [20]:
def flatten_sections(sections):

    flat = []

    for sec in sections:

        for el in sec["elements"]:

            flat.append({
                **el,
                "section_title": sec["section_title"],
                "section_id": sec["section_id"],
                "page_start": sec["page_start"],
                "page_end": sec["page_end"]
            })

    return flat


In [21]:
flat_elements = flatten_sections(sections)

print("Elementos planos:", len(flat_elements))


Elementos planos: 526


## Reconstrucción del contexto ampliado

Se implementa una función que permite recuperar el texto completo de una sección.

Esta funcionalidad será utilizada durante la fase de generación para proporcionar al modelo lenguaje el contexto completo relacionado con los fragmentos recuperados.


In [22]:
def reconstruct_section(section):

    return "\n".join(el["text"] for el in section["elements"])


## Persistencia del dataset estructural

Se guarda la representación jerárquica del documento en formato JSON.

Este archivo constituye la base documental del sistema RAG y permite:

- Reproducibilidad del pipeline
- Separación entre parsing e indexación
- Reutilización en procesos posteriores


In [23]:
with open("structured_sections.json", "w", encoding="utf-8") as f:
    json.dump(sections, f, ensure_ascii=False, indent=2)


In [24]:
len(sections)
sections[0]["section_title"]
sections[0]["elements"][:3]


[{'element_id': 'f3b8e9af-e788-4324-b34d-d75343674238',
  'type': 'UncategorizedText',
  'text': 'APROBADO POR EL CONSEJO NACIONAL DE CIBERSEGURIDAD EL DÍA 21 DE FEBRERO DE 2020',
  'page': 1,
  'source': 'guia_gestion_ciberincidentes .pdf',
  'char_start': 0,
  'char_end': 79}]